In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, StackingRegressor, StackingClassifier
from xgboost import XGBRegressor, XGBClassifier
from sklearn.linear_model import LogisticRegression, Ridge 
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.calibration import CalibratedClassifierCV
from arch import arch_model
from hmmlearn.hmm import GaussianHMM
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from hmmlearn.hmm import GaussianHMM
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import STL
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
import pykalman
import joblib
import gc
import MetaTrader5 as mt5
from datetime import datetime, timezone, timedelta
import time
import pytz
from threading import Event
import sys

def get_signal():
    ticker = 'XAUUSD_i'
    interval = mt5.TIMEFRAME_M15
    rates = mt5.copy_rates_from_pos(ticker, interval, 1, 777)
    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s')
    df.rename(columns={
        'time': 'Date',
        'open': 'Open',
        'high': 'High',
        'low': 'Low',
        'close': 'Close'
    }, inplace=True)
    df.set_index('Date', inplace=True)
    df = df.drop(columns=['tick_volume', 'real_volume', 'spread'])

    def process_features(df):

        # Calculate technical indicators
        df["returns"] = np.log(df.Close.div(df.Close.shift(1)))
        df["mom"] = df["returns"].rolling(21).mean()
        df["vol"] = df["returns"].rolling(21).std()
        df["max"] = df['Close'].rolling(21).max() / df['Close'] - 1
        df.dropna(inplace=True)

        # ARIMAX
        arimax_model = ARIMA(df['returns'].dropna(), exog=df[['vol', 'mom']], order=(5, 1, 5)).fit()
        df['arimax_pred'] = arimax_model.predict(
            start=1, 
            end=len(df)-1, 
            exog=df[['vol', 'mom']].iloc[1:len(df)]
        )
        df['arimax_fitted'] = arimax_model.fittedvalues
        df['arimax_residuals'] = arimax_model.resid
        df.dropna(inplace=True)

        # Kalman Filter
        kf = pykalman.KalmanFilter(initial_state_mean=df['returns'].iloc[0], n_dim_obs=1)
        state_means, state_covariances = kf.filter(df['returns'].values)
        df['kalman_estimate'] = state_means[:, 0]  # Extract the first column for estimated state
        df['kalman_residual'] = df['returns'] - df['kalman_estimate']  # Residual
        df['kalman_state_covariance'] = state_covariances[:, 0]  # Extracting one dimension of covariance
        df.dropna(inplace=True)
        gc.collect()

        # STL
        stl = STL(df['returns'], seasonal=13, period=5).fit()
        df['stl_trend'] = stl.trend
        df['stl_seasonal'] = stl.seasonal
        df['stl_resid'] = stl.resid
        df.dropna(inplace=True)
        gc.collect()

        # Final cleanup
        df.dropna(inplace=True)
        return df
    
    df = process_features(df)
    features = ['returns', 'kalman_residual', 'max', 'arimax_pred', 'arimax_residuals', 'stl_trend', 'stl_resid']
    X = df[features]
    objects = joblib.load('U2.joblib')
    scaler_reg = objects['scaler_clf']
    X = scaler_reg.transform(X)
    X = pd.DataFrame(X,columns=features)
    model = objects['stacking_clf']
    signal = model.predict(X)
    
    signal_time = df.index[-1]  # Latest timestamp from dataframe
    return signal[-1], signal_time

def get_open_position():
    positions = mt5.positions_get()
    if positions:
        return positions[0]  # Assuming only one position for simplicity
    return None

def close_position(position):
    ticket = position.ticket
    if position.type == mt5.ORDER_TYPE_BUY:
        close_action = mt5.ORDER_TYPE_SELL
        price = mt5.symbol_info_tick(position.symbol).bid
    else:
        close_action = mt5.ORDER_TYPE_BUY
        price = mt5.symbol_info_tick(position.symbol).ask
    
    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": position.symbol,
        "volume": position.volume,
        "type": close_action,
        "position": ticket,
        "price": price,
        "comment": "close the position",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_IOC,
    }
    
    result = mt5.order_send(request)
    print(f"Close order result: {result}")


def execute_trade(signal, qty):
    ticker = 'XAUUSD_i'
    if signal == 1:
        order_type = mt5.ORDER_TYPE_BUY
        price = mt5.symbol_info_tick(ticker).ask
        action = "BUY"
    elif signal == -1:
        order_type = mt5.ORDER_TYPE_SELL
        price = mt5.symbol_info_tick(ticker).bid
        action = "SELL"
    else:
        print("No action needed")
        return
    
    print(f"Executing {action} order: {ticker}, Volume: {qty}, Price: {price}")
    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": ticker,
        "volume": qty,
        "type": order_type,
        "price": price,
        "comment": "python open",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_IOC,
    }
    result = mt5.order_send(request)
    print(f"Trade order result: {result}")

def get_mt5_time():
    tz_mt5 = pytz.timezone('Etc/GMT-3')  # Use the correct timezone
    now = datetime.now(tz_mt5)
    return now.strftime('%Y-%m-%d %H:%M:%S')
    

def get_next_bar_time(interval):
    now = get_mt5_time()
    now = datetime.strptime(now, '%Y-%m-%d %H:%M:%S')
    if interval == mt5.TIMEFRAME_M15:
        minutes_past = now.minute % 15
        minutes_to_next_bar = (15 - minutes_past) % 15
        if minutes_to_next_bar == 0:
            minutes_to_next_bar = 15
        
        # Calculate the exact next bar time
        next_bar_time = now.replace(second=0, microsecond=0) + timedelta(minutes=minutes_to_next_bar)
        return next_bar_time
    else:
        raise ValueError("Unsupported timeframe")


def check_and_trade(stop_event):

    while not stop_event.is_set():
        try:         
            # Print mt5 time
            MT5 = get_mt5_time()
            MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
            print(f"Current MT5 time: {MT5}")
            
            # Get next bar time and check time
            next_bar_time = get_next_bar_time(mt5.TIMEFRAME_M15)  # Change to H4 timeframe
            next_check_time = next_bar_time + timedelta(seconds=0.1)
            print(f"Next check time: {next_check_time}")
            print(f"------------------------------------------------------------------------")

            # Wait until 3 seconds after the bar closes
            while MT5 < next_check_time:
                MT5 = get_mt5_time()
                MT5 = datetime.strptime(MT5, '%Y-%m-%d %H:%M:%S')
                time.sleep(0.1)  # Sleep briefly to avoid busy waiting
                       
            # Fetch current signal and its time
            current_signal, signal_time = get_signal()
            print(f"New signal checked: {current_signal}, Signal time: {signal_time}")

            # Fetch open position
            open_position = get_open_position()

            # Check if the signal time is 30 minutes behind current MT5 time
            time_diff = MT5 - signal_time
            if time_diff > timedelta(minutes= 30):
                print("We do not have a new signal yet")
                if open_position:
                    close_position(open_position)
                continue  # Skip further processing

            if open_position:
                if (current_signal == 1.0 and open_position.type == mt5.ORDER_TYPE_SELL) or \
                   (current_signal == -1.0 and open_position.type == mt5.ORDER_TYPE_BUY):
                    print(f"Current signal is {current_signal}.previous was opposite closing position.")
                    close_position(open_position)
                    # After closing, wait to ensure the position is closed before opening a new one
                    time.sleep(0.1)
                    open_position = get_open_position()  # Re-fetch the open position status
                    if open_position is None:
                        execute_trade(current_signal, 1.00)
                    else:
                        print("Failed to close the position. Not executing new trade.")
                else:
                    print(f"Position exists but the signal is the same or mismatched. No action needed.")
            else:
                if current_signal:
                    execute_trade(current_signal, 1.00)
                else:
                    print("No signal to act upon.")

        except Exception as e:
            print(f"An error occurred: {e}")
            time.sleep(60)  # Wait before retrying in case of error


# Initialize MetaTrader 5 connection and login
mt5.initialize()
username = int(os.environ['MT5_LOGIN'])
password = os.environ['MT5_PASSWORD']
server = 'Alpari-MT5-Demo'
mt5.login(username, password, server)

# Create a stop event
stop_event = Event()

try:
    # Start the trading loop
    check_and_trade(stop_event)
except KeyboardInterrupt:
    # Handle manual interruption
    print("Interrupted by user")
finally:
    # Shutdown MetaTrader 5 connection when done
    mt5.shutdown()
    print("MetaTrader 5 connection closed")